# Cybersecurity — Ransomware-like Burst Activity (Toy)

Model noisy file I/O with periodic encryption bursts. Detect burst periodicity and visualize with QFT peaks for interpretability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, rfftfreq
from quantum_hybrid_system import PeriodicState

rng = np.random.default_rng(6)
T = 8000
burst_period = 90
timeline = rng.poisson(0.05, size=T).astype(float)
# Inject bursts
for start in range(0, T, burst_period):
    if rng.random() > 0.2:
        timeline[start:start+10] += rng.poisson(2.5, size=min(10, T-start))

plt.figure()
plt.plot(timeline[:800])
plt.title("File I/O timeline (first 800 ticks)")
plt.xlabel("tick")
plt.ylabel("ops")
plt.show()

yf = np.abs(rfft(timeline - timeline.mean()))
xf = rfftfreq(T, d=1.0)
k = np.argmax(yf[1:]) + 1
freq = xf[k]
period_est = 1/freq
print("Estimated burst period ~", period_est)

n = 10
r = max(2, min(int(round(period_est)), 128))
ps = PeriodicState(num_qubits=n, period=r)
samples = ps.measure(num_shots=4000, use_qft=True)

bins = 64
hist = np.zeros(bins, dtype=int)
N = 2**n
for s in samples:
    hist[(s * bins) // N] += 1

plt.figure()
plt.bar(np.arange(bins), hist)
plt.title(f"QFT histogram (burst periodicity r≈{r})")
plt.xlabel("coarse bin")
plt.ylabel("counts")
plt.show()